# Människa-i-loopen Arbetsflöde med Microsoft Agent Framework

## 🎯 Inlärningsmål

I denna anteckningsbok kommer du att lära dig hur du implementerar **människa-i-loopen** arbetsflöden med Microsoft Agent Frameworks `RequestInfoExecutor`. Detta kraftfulla mönster gör att du kan pausa AI-arbetsflöden för att samla in mänsklig input, göra dina agenter interaktiva och ge människor kontroll över kritiska beslut.

## 🔄 Vad är Människa-i-Looper?

**Människa-i-loopen (HITL)** är ett designmönster där AI-agenter pausar körningen för att begära mänskligt input innan de fortsätter. Detta är avgörande för:

- ✅ **Kritiska beslut** - Få mänskligt godkännande innan viktiga åtgärder vidtas
- ✅ **Tvåtydiga situationer** - Låt människor klargöra när AI är osäker
- ✅ **Användarpreferenser** - Be användare välja mellan flera alternativ
- ✅ **Efterlevnad & säkerhet** - Säkerställ mänsklig övervakning för reglerade operationer
- ✅ **Interaktiva upplevelser** - Bygg konversationsagenter som svarar på användarinput

## 🏗️ Hur det fungerar i Microsoft Agent Framework

Frameworket tillhandahåller tre nyckelkomponenter för HITL:

1. **`RequestInfoExecutor`** - En särskild exekverare som pausar arbetsflödet och avger en `RequestInfoEvent`
2. **`RequestInfoMessage`** - Bas-klass för typade förfrågningspaket som skickas till människor
3. **`RequestResponse`** - Korrelerar mänskliga svar med ursprungliga förfrågningar med hjälp av `request_id`

**Arbetsflödesmönster:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Vårt exempel: Hotellbokning med användarbekräftelse

Vi bygger vidare på det villkorade arbetsflödet genom att lägga till mänsklig bekräftelse **innan** alternativa destinationer föreslås:

1. Användare begär en destination (t.ex. "Paris")
2. `availability_agent` kontrollerar om rum finns tillgängliga
3. **Om inga rum** → `confirmation_agent` frågar "Vill du se alternativ?"
4. Arbetsflödet **pausas** med `RequestInfoExecutor`
5. **Människan svarar** "ja" eller "nej" via konsolinput
6. `decision_manager` styr baserat på svaret:
   - **Ja** → Visa alternativa destinationer
   - **Nej** → Avbryt bokningsförfrågan
7. Visa slutresultatet

Detta visar hur man ger användare kontroll över agentens förslag!

---

Låt oss sätta igång! 🚀


## Steg 1: Importera nödvändiga bibliotek

Vi importerar standardkomponenterna för Agent Framework plus **klasser specifika för människa-i-loopen**:
- `RequestInfoExecutor` - Utförare som pausar arbetsflödet för mänsklig inmatning
- `RequestInfoEvent` - Händelse som avges när mänsklig inmatning efterfrågas
- `RequestInfoMessage` - Bas-klass för typade förfrågnings-payloads
- `RequestResponse` - Korrelerar mänskliga svar med förfrågningar
- `WorkflowOutputEvent` - Händelse för att upptäcka arbetsflödets utdata


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Steg 2: Definiera Pydantic-modeller för strukturerade utdata

Dessa modeller definierar **schemat** som agenter kommer att returnera. Vi behåller alla modeller från det villkorliga arbetsflödet och lägger till:

**Nytt för människa-i-loop:**
- `HumanFeedbackRequest` - Underklass av `RequestInfoMessage` som definierar förfrågningspayload som skickas till människor
  - Innehåller `prompt` (fråga att ställa) och `destination` (kontext om den otillgängliga staden)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Steg 3: Skapa hotellbokningsverktyget

Samma verktyg från det villkorliga arbetsflödet - kontrollerar om rum finns tillgängliga på destinationen.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Steg 4: Definiera villkorsfunktioner för dirigering

Vi behöver **fyra villkorsfunktioner** för vårt människa-i-loop-arbetsflöde:

**Från villkorsstyrt arbetsflöde:**
1. `has_availability_condition` - Dirigerar när hotell ÄR tillgängliga
2. `no_availability_condition` - Dirigerar när hotell INTE är tillgängliga

**Nytt för människa-i-loop:**
3. `user_wants_alternatives_condition` - Dirigerar när användaren säger "ja" till alternativ
4. `user_declines_alternatives_condition` - Dirigerar när användaren säger "nej" till alternativ


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Steg 5: Skapa Decision Manager Executor

Detta är **kärnan i human-in-the-loop-mönstret**! `DecisionManager` är en anpassad `Executor` som:

1. **Tar emot mänsklig feedback** via `RequestResponse`-objekt
2. **Bearbetar användarens beslut** (ja/nej)
3. **Styr arbetsflödet** genom att skicka meddelanden till lämpliga agenter

Viktiga funktioner:
- Använder `@handler`-dekoration för att exponera metoder som arbetsflödessteg
- Tar emot `RequestResponse[HumanFeedbackRequest, str]` som innehåller både ursprunglig förfrågan och användarens svar
- Genererar enkla "ja" eller "nej" meddelanden som triggar våra villkorsfunktioner


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Steg 6: Skapa en anpassad display-exekutor

Samma display-exekutor från villkorligt arbetsflöde - ger slutliga resultat som arbetsflödets utdata.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Steg 7: Ladda miljövariabler

Konfigurera LLM-klienten (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Steg 8: Skapa AI-agenter och exekutor

Vi skapar **sex arbetsflödeskomponenter**:

**Agenter (inkluderade i AgentExecutor):**
1. **availability_agent** - Kontrollerar hotellstatus med hjälp av verktyget
2. **confirmation_agent** - 🆕 Förbereder förfrågan om mänsklig bekräftelse
3. **alternative_agent** - Föreslår alternativa städer (när användaren säger ja)
4. **booking_agent** - Uppmuntrar bokning (när rum finns tillgängliga)
5. **cancellation_agent** - 🆕 Hanterar avbokningsmeddelande (när användaren säger nej)

**Speciella exekutor:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` som pausar arbetsflödet för mänskligt input
7. **decision_manager** - 🆕 Anpassad exekutor som styr baserat på mänskligt svar (redan definierad ovan)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Steg 9: Bygg arbetsflödet med mänsklig inblandning

Nu bygger vi arbetsflödesgrafen med **villkorlig dirigering** inklusive vägen för mänsklig inblandning:

**Arbetsflödets struktur:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Viktiga länkar:**
- `availability_agent → confirmation_agent` (när inga rum finns)
- `confirmation_agent → prepare_human_request` (omvandla typ)
- `prepare_human_request → request_info_executor` (paus för mänsklig handling)
- `request_info_executor → decision_manager` (alltid - tillhandahåller RequestResponse)
- `decision_manager → alternative_agent` (när användaren säger "ja")
- `decision_manager → cancellation_agent` (när användaren säger "nej")
- `availability_agent → booking_agent` (när rum finns tillgängliga)
- Alla vägar slutar vid `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Steg 10: Kör testfall 1 - Stad UTAN tillgänglighet (Paris med mänsklig bekräftelse)

Detta test visar den **fullständiga människan-i-loopen-cykeln**:

1. Begär hotell i Paris
2. availability_agent kontrollerar → Inga rum
3. confirmation_agent skapar fråga riktad till människa
4. request_info_executor **pausar arbetsflödet** och avger `RequestInfoEvent`
5. **Applikationen upptäcker händelsen och uppmanar användaren i konsolen**
6. Användaren skriver "ja" eller "nej"
7. Applikationen skickar svar via `send_responses_streaming()`
8. decision_manager styr baserat på svaret
9. Slutligt resultat visas

**Viktigt mönster:**
- Använd `workflow.run_stream()` för första iterationen
- Använd `workflow.send_responses_streaming(pending_responses)` för efterföljande iterationer
- Lyssna efter `RequestInfoEvent` för att upptäcka när mänskligt input behövs
- Lyssna efter `WorkflowOutputEvent` för att fånga slutgiltiga resultat


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Steg 11: Kör testfall 2 - Stad MED tillgänglighet (Stockholm - Ingen mänsklig inmatning behövs)

Detta test visar **den direkta vägen** när rum finns tillgängliga:

1. Begär hotell i Stockholm
2. availability_agent kontrollerar → Rum tillgängliga ✅
3. booking_agent föreslår bokning
4. display_result visar bekräftelse
5. **Ingen mänsklig inmatning krävs!**

Arbetsflödet kringgår helt den mänskliga processen när rum finns tillgängliga.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Viktiga punkter och bästa praxis för mänsklig i loopen

### ✅ Vad du har lärt dig:

#### 1. **RequestInfoExecutor-mönstret**
Mänsklig i loopen-mönstret i Microsoft Agent Framework använder tre nyckelkomponenter:
- `RequestInfoExecutor` - Pausar arbetsflödet och skickar ut händelser
- `RequestInfoMessage` - Bas-klass för typade förfrågningsdata (ärva från denna!)
- `RequestResponse` - Korrelerar mänskliga svar med ursprungliga förfrågningar

**Viktig insikt:**
- `RequestInfoExecutor` samlar INTE in input själv - den pausar bara arbetsflödet
- Din applikationskod måste lyssna efter `RequestInfoEvent` och samla in input
- Du måste anropa `send_responses_streaming()` med en dict som kopplar `request_id` till användarens svar

#### 2. **Streaming-exekveringsmönstret**
```python
# Första iterationen
stream = workflow.run_stream(initial_request)

# Efterföljande iterationer (efter mänsklig inmatning)
stream = workflow.send_responses_streaming(pending_responses)

# Bearbeta alltid händelser
events = [event async for event in stream]
```

#### 3. **Händelsestyrd arkitektur**
Lyssna efter specifika händelser för att styra arbetsflödet:
- `RequestInfoEvent` - Mänsklig input behövs (arbetsflödet pausat)
- `WorkflowOutputEvent` - Slutresultatet är tillgängligt (arbetsflödet klart)
- `WorkflowStatusEvent` - Statusändringar (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, etc.)

#### 4. **Anpassade executors med @handler**
`DecisionManager` visar hur man skapar executors som:
- Använder `@handler`-dekoration för att exponera metoder som arbetsflödets steg
- Tar emot typade meddelanden (t.ex. `RequestResponse[HumanFeedbackRequest, str]`)
- Rutar arbetsflödet genom att skicka meddelanden till andra executors
- Får tillgång till kontext via `WorkflowContext`

#### 5. **Villkorlig dirigering med mänskliga beslut**
Du kan skapa villkorliga funktioner som utvärderar mänskliga svar:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Verkliga tillämpningar:

1. **Godkännandearbetsflöden**
   - Få chefens godkännande innan utgiftsrapporter behandlas
   - Kräver mänsklig granskning innan automatiska e-post skickas
   - Bekräfta transaktioner med högt värde innan utförande

2. **Innehållsmoderering**
   - Markera tvivelaktigt innehåll för mänsklig granskning
   - Be moderatorer fatta slutgiltigt beslut i gråzoner
   - Eskalera till människor när AI:s säkerhet är låg

3. **Kundservice**
   - Låt AI hantera rutinfrågor automatiskt
   - Eskalera komplexa problem till mänskliga agenter
   - Fråga kunden om de vill prata med en människa

4. **Databehandling**
   - Be människor lösa tvetydiga datauppgifter
   - Bekräfta AI-tolkningar av oklara dokument
   - Låt användare välja mellan flera giltiga tolkningar

5. **Säkerhetskritiska system**
   - Kräver mänsklig bekräftelse innan oåterkalleliga åtgärder
   - Få godkännande innan tillgång till känslig data
   - Bekräfta beslut inom reglerade branscher (vård, finans)

6. **Interaktiva agenter**
   - Bygg konversationsbotar som ställer följdfrågor
   - Skapa guider som leder användare genom komplexa processer
   - Designa agenter som samarbetar med människor steg för steg

### 🔄 Jämförelse: Med vs utan mänsklig i loopen

| Funktion | Villkorligt arbetsflöde | Arbetsflöde med mänsklig i loopen |
|---------|---------------------|---------------------------|
| **Exekvering** | Enkel `workflow.run()` | Loop med `run_stream()` + `send_responses_streaming()` |
| **Användarinput** | Ingen (fullt automatiserad) | Interaktiva uppmaningar via `input()` eller UI |
| **Komponenter** | Agenter + Executers | + RequestInfoExecutor + DecisionManager |
| **Händelser** | Endast AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent, etc. |
| **Pausning** | Ingen pausning | Arbetsflödet pausar vid RequestInfoExecutor |
| **Mänsklig kontroll** | Ingen mänsklig kontroll | Människor fattar viktiga beslut |
| **Användningsfall** | Automatisering | Samarbete och tillsyn |

### 🚀 Avancerade mönster:

#### Flera mänskliga beslutspunkter
Du kan ha flera `RequestInfoExecutor`-noder i samma arbetsflöde:
```python
.add_edge(agent1, request_info_1)  # Första mänskliga beslutet
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Andra mänskliga beslutet
.add_edge(decision_manager_2, final_agent)
```

#### Timeout-hantering
Implementera timeout för mänskliga svar:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Standardinställning till säker option
```

#### Rikt UI-integrering
Istället för `input()`, integrera med webb-UI, Slack, Teams, etc.:
```python
if isinstance(event, RequestInfoEvent):
    # Skicka meddelande till användarens föredragna kanal
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Villkorlig mänsklig i loopen
Be bara om mänsklig input i specifika situationer:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Rutt endast till människa om förtroendet är lågt eller värdet är högt
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Bästa praxis:

1. **Ärv alltid från RequestInfoMessage**
   - Ger typ-säkerhet och validering
   - Möjliggör rik kontext för UI-rendering
   - Klargör avsikten för varje förfrågningstyp

2. **Använd beskrivande uppmaningar**
   - Inkludera kontext om vad du frågar efter
   - Förklara konsekvenser av varje val
   - Håll frågor enkla och tydliga

3. **Hantera oväntad input**
   - Validera användarens svar
   - Ge standardvärden för ogiltig input
   - Ge tydliga felmeddelanden

4. **Spåra request IDs**
   - Använd korrelationen mellan request_id och svar
   - Försök inte hantera tillstånd manuellt

5. **Designa för icke-blockering**
   - Blockera inte trådar som väntar på input
   - Använd asynkrona mönster genomgående
   - Stöd samtidiga arbetsflödesinstanser

### 📚 Relaterade koncept:

- **Agent Middleware** - Avbryt agentanrop (tidigare anteckningsbok)
- **Arbetsflödesstatushantering** - Persistens av arbetsflödesstatus mellan körningar
- **Multi-Agent-samarbete** - Kombinera mänsklig i loopen med agentteam
- **Händelsestyrda arkitekturer** - Bygg reaktiva system med händelser

---

### 🎓 Grattis!

Du har behärskat arbetsflöden med mänsklig i loopen i Microsoft Agent Framework! Du vet nu hur du:
- ✅ Pausar arbetsflöden för att samla in mänsklig input
- ✅ Använder RequestInfoExecutor och RequestInfoMessage
- ✅ Hanterar streaming-exekvering med händelser
- ✅ Skapar anpassade executors med @handler
- ✅ Dirigerar arbetsflöden baserat på mänskliga beslut
- ✅ Bygger interaktiva AI-agenter som samarbetar med människor

**Detta är ett kritiskt mönster för att bygga pålitliga, kontrollerbara AI-system!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
